In [ ]:
import torch

from dataset_loaders import build_data_loaders
from utils.checkpoints import load_ae_from_path
from utils.wandb_utils import load_from_wandb
from utils.config import DatasetConfig
from utils.visualisation import plot_latent_space, show, show_comparison

In [ ]:
ae_path = load_from_wandb("variational_celeba")
ae = load_ae_from_path(ae_path, device=torch.device("mps"))

In [ ]:
dataset_cfg = DatasetConfig(
    name="celeba_single_attribute",
    channels=3,
    height=64,
    width=64,
    num_classes=2,
)

dataloader, test_loader = build_data_loaders(dataset_cfg, batch_size=64, shuffle_test=True, num_workers=0)


In [ ]:
images, labels = next(iter(test_loader))

with torch.no_grad():
    outputs = ae(images)
    recon = torch.sigmoid(outputs.reconstructed)

show_comparison(images, recon)


In [ ]:
from models.autoencoder import AbstractAutoencoder


def ae_latent_space(model: AbstractAutoencoder, data_loader: torch.utils.data.DataLoader,
                    title="Latent Space of CelebA Autoencoder"):
    samples = []
    sampled_labels = []
    i = 0
    for image, label in data_loader:
        latent = model.encode(image)
        samples.append(latent)
        sampled_labels.append(label)
        i = i + 1
        if i > 10:
            break

    samples = torch.cat(samples, dim=0)
    sampled_labels = torch.cat(sampled_labels, dim=0)
    plot_latent_space(samples, sampled_labels, title=title)



In [ ]:
def show_with_label(model: AbstractAutoencoder, data_loader: torch.utils.data.DataLoader):
    originals = []
    labels = []
    recons = []
    image, label = next(iter(data_loader))
    latent = model.encode(image)
    recon = model.decode(latent)
    originals.append(image)
    labels.append(label)
    recons.append(recon)

    originals = torch.stack(originals)
    labels = torch.stack(labels)
    recons = torch.stack(recons)
    show_comparison(originals[labels == 0], recons[labels == 0], "Label 0")
    show_comparison(originals[labels == 1], recons[labels == 1], "Label 1")


In [ ]:
show_with_label(ae, test_loader)

In [ ]:
ae_latent_space(ae, test_loader)

In [ ]:
random_latent = torch.rand(16, 128)
with torch.no_grad():
    random_images = ae.decode(random_latent)

show(random_images, title="Samples from random latents")

In [ ]:
versions = ["v1", "v2", "v3", "v4"]
images, _ = next(iter(test_loader))
images = images[:8]
for version in versions:
    ae_path = load_from_wandb(f"variational_celeba", tag=version)
    ae = load_ae_from_path(ae_path, device=torch.device("mps"))
    ae.eval()
    logits = ae(images)
    recon = torch.sigmoid(logits.reconstructed)
    show_comparison(images, recon, title=f"Autoencoder version {version}")
